# 🏗️ 빅데이터 플랫폼 요구사항 분석
## - 대구시 교통·환경·생활 통합 데이터 플랫폼

**학습 목표:**
1. 여러 공공 API에서 데이터를 수집하고 합쳐본다
2. 데이터를 보면서 **어떤 플랫폼이 필요한지** 요구사항을 도출한다
3. 다음 단계(ETL 파이프라인)로 이어지는 아키텍처를 이해한다

---
### 🎯 우리가 만들 플랫폼의 목표
> **"대구 시민이 버스를 기다리는 동안, 주변 날씨·공기질·편의시설을 한눈에 볼 수 있는 서비스"**

```
📡 데이터 수집 대상
├── 🚌 대구버스 정보시스템   → 버스 노선, 정류장, 실시간 위치
├── 🌤️ 기상청 단기예보       → 기온, 강수, 날씨
├── 🌫️ 한국환경공단 대기오염 → PM2.5, PM10, 오존
└── 🏥 건강보험심사평가원    → 주변 병원/약국
```

In [ ]:
# ✅ 공통 설정
import requests
import json
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from datetime import datetime
import time

API_KEY = "여기에_API키_입력".strip()

# 대구 중심 좌표
DAEGU_LAT, DAEGU_LNG = 35.8714, 128.6014

def safe_get(url, params, label=""):
    """안전한 API 호출 (에러 처리 포함)"""
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200:
            print(f"✅ {label} 수집 성공")
            return r.json()
        else:
            print(f"🔴 {label} 실패: {r.status_code}")
            return None
    except Exception as e:
        print(f"🔴 {label} 오류: {e}")
        return None

print("설정 완료! ✅")

---
## 📡 Step 1. 데이터 수집 - 4개 API에서 동시에 가져오기

### 💡 왜 여러 데이터를 합치나요?
각각의 API는 한 가지 정보만 줍니다.  
**진짜 가치 있는 서비스**는 여러 데이터를 연결할 때 생깁니다.

```
버스 정류장 위치  +  주변 날씨  +  공기질  +  병원  =  "지금 외출해도 될까?" 서비스
```

In [ ]:
# 🚌 [1] 대구버스 정류장 + 노선 수집
print("=" * 50)
print("[1/4] 대구버스 데이터 수집 중...")

bus_data = safe_get(
    url="https://apis.data.go.kr/6270000/dbmsapi02/getBasic02",
    params={"serviceKey": API_KEY},
    label="대구버스"
)

if bus_data:
    stops = bus_data["body"]["items"]["bs"]
    routes = bus_data["body"]["items"]["route"]
    df_stops = pd.DataFrame(stops)
    df_routes = pd.DataFrame(routes)
    print(f"   정류장: {len(df_stops)}개 | 노선: {len(df_routes)}개")

In [ ]:
# 🌤️ [2] 기상청 현재 날씨 수집
print("[2/4] 기상청 날씨 데이터 수집 중...")

now = datetime.now()
weather_data = safe_get(
    url="https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst",
    params={
        "serviceKey": API_KEY,
        "numOfRows": "10",
        "pageNo": "1",
        "dataType": "JSON",
        "base_date": now.strftime("%Y%m%d"),
        "base_time": now.strftime("%H00"),
        "nx": "89", "ny": "90"   # 대구
    },
    label="기상청 날씨"
)

if weather_data:
    weather_items = weather_data["response"]["body"]["items"]["item"]
    df_weather = pd.DataFrame(weather_items)
    # 카테고리별로 정리
    weather_dict = {row["category"]: row["obsrValue"] for _, row in df_weather.iterrows()}
    print(f"   기온: {weather_dict.get('T1H','?')}℃ | 습도: {weather_dict.get('REH','?')}% | 풍속: {weather_dict.get('WSD','?')}m/s")

In [ ]:
# 🌫️ [3] 한국환경공단 대기오염 정보
# 신청 URL: https://www.data.go.kr/data/15073861/openapi.do
print("[3/4] 대기오염 데이터 수집 중...")

air_data = safe_get(
    url="https://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getCtprvnRltmMesureDnsty",
    params={
        "serviceKey": API_KEY,
        "returnType": "json",
        "numOfRows": "20",
        "pageNo": "1",
        "sidoName": "대구",
        "ver": "1.0"
    },
    label="대기오염"
)

if air_data:
    air_items = air_data["response"]["body"]["items"]
    df_air = pd.DataFrame(air_items)
    print(f"   측정소 수: {len(df_air)}개")
    print(f"   컬럼: {df_air.columns.tolist()}")

In [ ]:
# 🏥 [4] 대구 병원 정보
print("[4/4] 병원 데이터 수집 중...")

hosp_data = safe_get(
    url="https://apis.data.go.kr/B551182/hospInfoServicev2/getHospBasisList",
    params={
        "serviceKey": API_KEY,
        "numOfRows": "100",
        "pageNo": "1",
        "sidoCd": "410000",
        "_type": "json"
    },
    label="병원"
)

if hosp_data:
    hosp_items = hosp_data["response"]["body"]["items"]["item"]
    df_hosp = pd.DataFrame(hosp_items)
    print(f"   병원 수: {len(df_hosp)}개")

print("\n" + "="*50)
print("✅ 4개 API 수집 완료!")

---
## 🔗 Step 2. 데이터 통합 - pandas merge

### 💡 연결 기준: **위치(좌표)**
서로 다른 API 데이터를 연결하는 핵심 키는 **위도/경도**입니다.

```
버스정류장 (bsId, bsNm, xPos, yPos)
        ↓  좌표 기반 반경 검색
병원     (yadmNm, XPos, YPos)
        ↓  측정소명 → 지역 매핑
대기오염  (stationName, pm10Value, pm25Value)
        ↓  격자 좌표 → 전체 공통 적용
날씨     (T1H, REH, WSD, PTY)
```

In [ ]:
# 🔗 버스 정류장 + 날씨 합치기
# 날씨는 대구 전체에 동일하게 적용 (격자 단위)

if bus_data and weather_data:
    # 날씨 정보를 딕셔너리로
    weather_dict = {row["category"]: row["obsrValue"] for _, row in df_weather.iterrows()}
    
    # 버스 정류장에 날씨 컬럼 추가
    df_stops["기온"] = weather_dict.get("T1H", None)
    df_stops["습도"] = weather_dict.get("REH", None)
    df_stops["풍속"] = weather_dict.get("WSD", None)
    df_stops["강수형태"] = weather_dict.get("PTY", "0")
    df_stops["수집시각"] = now.strftime("%Y-%m-%d %H:%M")
    
    print("버스 정류장 + 날씨 병합 결과:")
    print(df_stops[["bsNm", "xPos", "yPos", "기온", "습도", "강수형태"]].head())

In [ ]:
# 🔗 버스 정류장 + 대기오염 합치기
# 가장 가까운 측정소 기준으로 매핑
import math

def nearest_station(lat, lng, df_stations):
    """가장 가까운 대기 측정소 찾기"""
    min_dist = float('inf')
    nearest = None
    for _, row in df_stations.iterrows():
        try:
            dlat = float(row.get('dmX', 0) or 0) - lat
            dlng = float(row.get('dmY', 0) or 0) - lng
            dist = math.sqrt(dlat**2 + dlng**2)
            if dist < min_dist:
                min_dist = dist
                nearest = row
        except:
            pass
    return nearest

if air_data and bus_data:
    # 대기오염 데이터에 좌표 있는 것만
    df_air_loc = df_air[df_air['dmX'].notna()].copy()
    
    # 첫 5개 정류장에 대기오염 매핑 (시연용)
    results = []
    for _, stop in df_stops.head(5).iterrows():
        station = nearest_station(float(stop['yPos']), float(stop['xPos']), df_air_loc)
        if station is not None:
            results.append({
                "정류장": stop['bsNm'],
                "측정소": station.get('stationName', ''),
                "PM10": station.get('pm10Value', '-'),
                "PM2.5": station.get('pm25Value', '-'),
                "통합지수": station.get('khaiValue', '-')
            })
    
    df_merged_air = pd.DataFrame(results)
    print("정류장 + 대기오염 매핑 결과:")
    print(df_merged_air)

In [ ]:
# 🔗 정류장 반경 내 병원 수 집계
def count_nearby(lat, lng, df_target, lat_col, lng_col, radius_km=0.5):
    """반경 내 개수 세기"""
    R = 6371
    count = 0
    for _, row in df_target.iterrows():
        try:
            tlat, tlng = float(row[lat_col]), float(row[lng_col])
            dlat = math.radians(tlat - lat)
            dlng = math.radians(tlng - lng)
            a = math.sin(dlat/2)**2 + math.cos(math.radians(lat)) * math.cos(math.radians(tlat)) * math.sin(dlng/2)**2
            if R * 2 * math.asin(math.sqrt(a)) <= radius_km:
                count += 1
        except:
            pass
    return count

if hosp_data and bus_data:
    print("정류장 반경 500m 내 병원 수 계산 중... (시간 걸릴 수 있음)")
    sample_stops = df_stops.head(10).copy()
    
    sample_stops["주변병원수"] = sample_stops.apply(
        lambda row: count_nearby(
            float(row['yPos']), float(row['xPos']),
            df_hosp, 'YPos', 'XPos', radius_km=0.5
        ), axis=1
    )
    
    print("\n정류장 + 주변 병원 수 결과:")
    print(sample_stops[["bsNm", "주변병원수"]].sort_values("주변병원수", ascending=False))

---
## 📋 Step 3. 요구사항 도출

데이터를 직접 다뤄보니까 어떤 문제가 보이나요?  
이게 바로 **플랫폼이 필요한 이유**입니다!

### 🔍 데이터 탐색 중 발견되는 문제들

In [ ]:
# 📊 문제 1: 데이터마다 갱신 주기가 다르다
update_cycles = {
    "API": ["버스 실시간", "날씨(초단기)", "대기오염", "병원정보", "버스 기초정보"],
    "갱신주기": ["실시간(1분)", "매시간", "1시간", "비정기", "1일 1회"],
    "데이터양": ["중간", "소", "소", "대", "대"],
    "저장방식": ["시계열DB 필요", "덮어쓰기", "덮어쓰기", "정적저장", "정적저장"]
}
df_cycle = pd.DataFrame(update_cycles)
print("📋 데이터 갱신 주기 분석")
print(df_cycle.to_string(index=False))

In [ ]:
# 📊 문제 2: 좌표 체계가 다르다
coord_systems = {
    "API": ["대구버스", "기상청", "한국환경공단", "건강보험심사평가원"],
    "좌표체계": ["WGS84 (위경도)", "격자좌표 (nx,ny)", "WGS84 (위경도)", "WGS84 (위경도)"],
    "변환필요": ["X", "O (격자→위경도)", "X", "X"]
}
df_coord = pd.DataFrame(coord_systems)
print("📋 좌표 체계 분석 → 변환 로직이 필요!")
print(df_coord.to_string(index=False))

In [ ]:
# 📊 문제 3: 응답 구조가 다 다르다
response_structures = {
    "API": ["대구버스", "기상청", "한국환경공단", "건강보험심사평가원"],
    "응답구조": [
        "body.items.bs[]",
        "response.body.items.item[]",
        "response.body.items[]",
        "response.body.items.item[]"
    ],
    "결측값처리": ["없음", "코드값 변환 필요", "'-' 문자열", "null 혼재"]
}
df_struct = pd.DataFrame(response_structures)
print("📋 응답 구조 분석 → 파싱 로직 통일 필요!")
print(df_struct.to_string(index=False))

In [ ]:
# ✅ 요구사항 정리
requirements = """
╔══════════════════════════════════════════════════════╗
║         빅데이터 플랫폼 요구사항 (도출 결과)           ║
╠══════════════════════════════════════════════════════╣
║  [기능 요구사항]                                      ║
║  FR-01. 4개 API 데이터를 주기적으로 자동 수집          ║
║  FR-02. 좌표 체계 변환 (기상청 격자 ↔ WGS84)          ║
║  FR-03. 결측값/이상값 전처리 및 정제                  ║
║  FR-04. 정류장 기준 공간 데이터 통합 (반경 검색)        ║
║  FR-05. 수집 이력 저장 (시계열 분석용)                ║
║  FR-06. 통합 데이터 조회 API 제공                     ║
║                                                      ║
║  [비기능 요구사항]                                    ║
║  NFR-01. 실시간 데이터 1분 이내 갱신                  ║
║  NFR-02. API 장애 시 재시도 및 알림                   ║
║  NFR-03. 일 100만 건 이상 처리 가능                   ║
║  NFR-04. 데이터 90일 이상 보관                        ║
╚══════════════════════════════════════════════════════╝
"""
print(requirements)

---
## 🗺️ Step 4. 통합 데이터 지도 시각화
수집한 4개 데이터를 하나의 지도에 올려봅니다.

In [ ]:
# 🗺️ 4개 데이터 통합 지도
m = folium.Map(location=[DAEGU_LAT, DAEGU_LNG], zoom_start=12)

# 레이어 그룹 생성
layer_bus   = folium.FeatureGroup(name="🚌 버스 정류장")
layer_hosp  = folium.FeatureGroup(name="🏥 병원")
layer_air   = folium.FeatureGroup(name="🌫️ 대기 측정소")

# 버스 정류장 (파란 점)
if bus_data:
    bus_cluster = MarkerCluster()
    for stop in stops[:200]:
        folium.CircleMarker(
            location=[stop["yPos"], stop["xPos"]],
            radius=4, color="#3498db", fill=True,
            popup=stop["bsNm"], tooltip=stop["bsNm"]
        ).add_to(layer_bus)

# 병원 (빨간 플러스)
if hosp_data:
    for _, row in df_hosp.iterrows():
        try:
            folium.Marker(
                location=[float(row["YPos"]), float(row["XPos"])],
                popup=f"{row['yadmNm']} ({row.get('clCdNm','')})",
                icon=folium.Icon(color="red", icon="plus-sign", prefix="glyphicon")
            ).add_to(layer_hosp)
        except:
            pass

# 대기 측정소 (초록 구름)
if air_data:
    for _, row in df_air.iterrows():
        try:
            pm10 = row.get('pm10Value', '-')
            pm25 = row.get('pm25Value', '-')
            folium.Marker(
                location=[float(row['dmX']), float(row['dmY'])],
                popup=f"측정소: {row['stationName']}\nPM10: {pm10}\nPM2.5: {pm25}",
                icon=folium.Icon(color="green", icon="cloud", prefix="glyphicon")
            ).add_to(layer_air)
        except:
            pass

# 날씨 정보 (중심에 팝업)
if weather_data:
    weather_html = f"""
    <div style='font-family:sans-serif; padding:10px; min-width:150px'>
        <b>🌤️ 현재 날씨 ({now.strftime('%H:%M')})</b><br>
        기온: {weather_dict.get('T1H','?')}℃<br>
        습도: {weather_dict.get('REH','?')}%<br>
        풍속: {weather_dict.get('WSD','?')}m/s
    </div>
    """
    folium.Marker(
        location=[DAEGU_LAT, DAEGU_LNG],
        popup=folium.Popup(weather_html, max_width=200),
        icon=folium.Icon(color="orange", icon="sun", prefix="fa")
    ).add_to(m)

# 레이어 지도에 추가
layer_bus.add_to(m)
layer_hosp.add_to(m)
layer_air.add_to(m)
folium.LayerControl().add_to(m)  # 레이어 on/off 컨트롤

m.save("integrated_map.html")
print("✅ 4개 데이터 통합 지도 완성: integrated_map.html")
m

---
## 🏗️ Step 5. 아키텍처 설계 논의

지금까지 직접 겪은 문제들을 바탕으로 플랫폼 구조를 생각해봅시다.

### ❓ 토론 질문

**Q1.** 지금 코드의 문제점은 무엇인가요?
```
→ 실행할 때마다 API 호출 (트래픽 낭비, 속도 느림)
→ 데이터가 저장되지 않음 (이력 없음)
→ 오류나면 그냥 멈춤
→ 스케줄링이 없음 (자동 수집 불가)
```

**Q2.** 어떻게 해결하면 좋을까요?
```
→ 수집: 스케줄러로 자동화 (Airflow, cron)
→ 저장: 데이터베이스에 적재 (PostgreSQL, MongoDB)
→ 서빙: 저장된 데이터를 API로 제공 (FastAPI)
```

**이게 바로 다음 수업(ETL 파이프라인)에서 만들 것입니다!** 🚀

In [ ]:
# 📁 지금 수집한 데이터를 CSV로 저장 (다음 수업에서 사용!)
if bus_data:
    df_stops.to_csv("stops_with_weather.csv", index=False, encoding="utf-8-sig")
    df_routes.to_csv("bus_routes.csv", index=False, encoding="utf-8-sig")
    print("✅ stops_with_weather.csv 저장")
    print("✅ bus_routes.csv 저장")

if hosp_data:
    df_hosp.to_csv("hospitals_daegu.csv", index=False, encoding="utf-8-sig")
    print("✅ hospitals_daegu.csv 저장")

if air_data:
    df_air.to_csv("air_quality_daegu.csv", index=False, encoding="utf-8-sig")
    print("✅ air_quality_daegu.csv 저장")

print("\n💡 이 CSV 파일들이 다음 ETL 수업의 원천 데이터가 됩니다!")

---
## 📝 오늘 수업 정리

### 배운 것
| 단계 | 내용 | 핵심 코드 |
|------|------|----------|
| 수집 | 4개 API 동시 호출 | `safe_get()` + `requests.get()` |
| 통합 | 좌표 기반 연결 | `count_nearby()` + 반경 계산 |
| 분석 | 데이터 구조 차이 파악 | `pd.DataFrame()` + 비교 |
| 시각화 | 레이어별 통합 지도 | `folium.FeatureGroup` |
| 저장 | CSV 내보내기 | `df.to_csv()` |

### 발견한 문제 → 다음 수업 연결
```
❌ 지금                    ✅ 다음 수업 (ETL 파이프라인)
──────────────────────────────────────────────────
매번 수직 호출        →   Airflow 스케줄러 자동화
메모리에만 존재       →   PostgreSQL DB 적재
좌표 변환 매번 반복   →   변환 함수 모듈화
오류시 중단           →   재시도 + 로그 저장
CSV 파일              →   구조화된 테이블 설계
```